In [ ]:
# --- SETUP ---
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import pandas as pd
import sqlalchemy as sqla
from sqlalchemy import text
import urllib
import pyodbc


# --- CONNECTION INFO ---
server = r"xxxxxx"   
database = "xxxxxx"
username = "xxxxxx"
password = "xxxxxx"                

# This connection string works with pyodbc.connect() — so we reuse it for SQLAlchemy
conn_str = (
    "DRIVER={ODBC Driver 18 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
    "Connection Timeout=5;"
)

# --- 1. QUICK TEST: MAKE SURE PYODBC CONNECTS ---
try:
    test_conn = pyodbc.connect(conn_str)
    cur = test_conn.cursor()
    print("Connected to SQL server")
    cur.execute("SELECT 1")
    print(cur.fetchone())   # (1,)
    test_conn.close()
except Exception as e:
    print("Connection Failed:", e)


# --- 2. CREATE SQLALCHEMY ENGINE USING THE SAME WORKING CONNECTION ---
def get_conn():
    return pyodbc.connect(conn_str)

engine = sqla.create_engine(
    "mssql+pyodbc://",
    creator=get_conn
)

Connected to SQL server
(1,)


In [2]:
# --- 3. CREATE SCHEMA `final_project_` ---
schema_cmd = text("CREATE SCHEMA final_project3")

try:
    with engine.begin() as conn:   # begins transaction + auto-commits
        conn.execute(schema_cmd)
    print("Schema created successfully.")
except Exception as e:
    print("Schema creation failed:", e)

Schema created successfully.


In [3]:
# Creating tables (DDL) **AI helped to validate datatypes**
ddl = """
---------------------------------------------------------------------
-- final_project3 schema ()
---------------------------------------------------------------------

CREATE TABLE final_project3.Weather (
    Weather_ID      VARCHAR(5)   NOT NULL PRIMARY KEY,
    Weather         VARCHAR(100) NOT NULL,
    Road_Conditions VARCHAR(100) NOT NULL
);

CREATE TABLE final_project3.Road (
    Road_ID      VARCHAR(5)   NOT NULL PRIMARY KEY,
    Road_Type    VARCHAR(100) NOT NULL,
    Road_Surface VARCHAR(100) NOT NULL
);

CREATE TABLE final_project3.Crash (
    Crash_ID    VARCHAR(5)   NOT NULL PRIMARY KEY,
    Crash_Type  VARCHAR(100) NOT NULL,
    [Date]      DATETIME NULL,          -- allows NaT / NULL from pandas
    Temperature DECIMAL(4,1) NULL,      -- pandas invalid → NULL
    Location    VARCHAR(50) NULL,       -- NaN found in location
    Road_ID     VARCHAR(5)   NOT NULL,
    Weather_ID  VARCHAR(5)   NOT NULL,
    CONSTRAINT FK_Crash_Road
        FOREIGN KEY (Road_ID) REFERENCES final_project3.Road(Road_ID),
    CONSTRAINT FK_Crash_Weather
        FOREIGN KEY (Weather_ID) REFERENCES final_project3.Weather(Weather_ID)
);

CREATE TABLE final_project3.Vehicle (
    Vehicle_ID   VARCHAR(5)  NOT NULL PRIMARY KEY,
    Vehicle_Type VARCHAR(60) NOT NULL,
    Crash_ID     VARCHAR(5)  NOT NULL,
    CONSTRAINT FK_Vehicle_Crash
        FOREIGN KEY (Crash_ID) REFERENCES final_project3.Crash(Crash_ID)
);

CREATE TABLE final_project3.Person (
    PersonID    VARCHAR(5)   NOT NULL PRIMARY KEY,
    Age         VARCHAR(5) NULL,                       -- NaN allowed
    Gender      VARCHAR(20)  NULL,                  -- NaN allowed
    Injury_Area VARCHAR(100) NULL,                  -- NaN allowed
    Injured     VARCHAR(50)  NOT NULL,
    Injury      VARCHAR(50)  NOT NULL,
    Vehicle_ID  VARCHAR(5)   NOT NULL,
    CONSTRAINT FK_Person_Vehicle
        FOREIGN KEY (Vehicle_ID) REFERENCES final_project3.Vehicle(Vehicle_ID)
);
"""

try:
    with engine.begin() as conn:
        conn.exec_driver_sql(ddl)
    print("All tables created successfully in final_project3.")
except Exception as e:
    print("Table creation failed:", e)


All tables created successfully in final_project3.


In [5]:
# Loading the raw traffic injury dataset from Excel into a DataFrame
df = pd.read_excel(r"C:\Users\Nida Aslam\Downloads\traffic_injury_edited.xlsx")
df.head()

,AGE,LOCATION,CRASHDATE,GENDER,INJURY,CRASH TYPE,ROAD CONDTIONS,ROAD TYPE,ROAD SURFACE,VEHICLE TYPE,INJURED,WEATHER,TEMPERATURE_F,INJURY_AREA
0,66,PLEASANT RIDGE,2022-01-01 00:01:00,MALE,MINOR,ANGLE,WET,CURVE GRADE,"BLACKTOP, BITUMINOUS, ASPHALT",PASSENGER CAR,DRIVER,RAIN,52.8,Abdomen - Pelvis
1,49,CAMP WASHINGTON,2022-01-01 00:37:00,MALE,MINOR,UNKNOWN,WET,CURVE LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",SPORT UTILITY VEHICLE,DRIVER,RAIN,52.8,Abdomen - Pelvis; Neck
2,9,CAMP WASHINGTON,2022-01-01 00:37:00,MALE,MINOR,UNKNOWN,WET,CURVE LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",SPORT UTILITY VEHICLE,OCCUPANT,RAIN,52.8,Back
3,39,CAMP WASHINGTON,2022-01-01 00:37:00,FEMALE,MINOR,UNKNOWN,WET,CURVE LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",SPORT UTILITY VEHICLE,OCCUPANT,RAIN,52.8,Does Not Apply
4,38,MOUNT AUBURN,2022-01-01 02:44:00,MALE,MINOR,NOT COLLISION BETWEEN TWO MOTOR VEHICLES IN TR...,WET,CURVE GRADE,"BLACKTOP, BITUMINOUS, ASPHALT",PASSENGER CAR,DRIVER,RAIN,35.4,Neck; Face; Does Not Apply


In [6]:
# Standardize column names for easier referencing, then preview the data and check each column’s data type
df.columns = df.columns.str.lower().str.replace(" ", "_")
df.head()
df.dtypes

,age,location,crashdate,gender,injury,crash_type,road_condtions,road_type,road_surface,vehicle_type,injured,weather,temperature_f,injury_area
0,66,PLEASANT RIDGE,2022-01-01 00:01:00,MALE,MINOR,ANGLE,WET,CURVE GRADE,"BLACKTOP, BITUMINOUS, ASPHALT",PASSENGER CAR,DRIVER,RAIN,52.8,Abdomen - Pelvis
1,49,CAMP WASHINGTON,2022-01-01 00:37:00,MALE,MINOR,UNKNOWN,WET,CURVE LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",SPORT UTILITY VEHICLE,DRIVER,RAIN,52.8,Abdomen - Pelvis; Neck
2,9,CAMP WASHINGTON,2022-01-01 00:37:00,MALE,MINOR,UNKNOWN,WET,CURVE LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",SPORT UTILITY VEHICLE,OCCUPANT,RAIN,52.8,Back
3,39,CAMP WASHINGTON,2022-01-01 00:37:00,FEMALE,MINOR,UNKNOWN,WET,CURVE LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",SPORT UTILITY VEHICLE,OCCUPANT,RAIN,52.8,Does Not Apply
4,38,MOUNT AUBURN,2022-01-01 02:44:00,MALE,MINOR,NOT COLLISION BETWEEN TWO MOTOR VEHICLES IN TR...,WET,CURVE GRADE,"BLACKTOP, BITUMINOUS, ASPHALT",PASSENGER CAR,DRIVER,RAIN,35.4,Neck; Face; Does Not Apply


age                       object
location                  object
crashdate         datetime64[ns]
gender                    object
injury                    object
crash_type                object
road_condtions            object
road_type                 object
road_surface              object
vehicle_type              object
injured                   object
weather                   object
temperature_f            float64
injury_area               object
dtype: object

In [7]:
# Check how many missing values each column has and display sample rows containing any nulls
df.isna().sum()
df[df.isna().any(axis=1)].head()

age                21
location          184
crashdate           0
gender             15
injury              0
crash_type          0
road_condtions      0
road_type           0
road_surface        0
vehicle_type        0
injured             0
weather             0
temperature_f       0
injury_area       110
dtype: int64

,age,location,crashdate,gender,injury,crash_type,road_condtions,road_type,road_surface,vehicle_type,injured,weather,temperature_f,injury_area
44,71,NaN,2022-01-10 07:38:00,FEMALE,MINOR,NOT COLLISION BETWEEN TWO MOTOR VEHICLES IN TR...,DRY,STRAIGHT LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",PEDESTRIAN/SKATER,PEDESTRIAN,CLEAR,89.2,Eye; Elbow-Lower-Arm-Hand; Shoulder - Upper Arm
167,44,NaN,2022-01-28 18:25:00,FEMALE,MINOR,NOT COLLISION BETWEEN TWO MOTOR VEHICLES IN TR...,DRY,STRAIGHT LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",PASSENGER CAR,DRIVER,CLEAR,37.9,Shoulder - Upper Arm; Face; Elbow-Lower-Arm-Hand
184,22,NaN,2022-01-29 18:50:00,FEMALE,MINOR,ANGLE,DRY,STRAIGHT LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",PASSENGER CAR,OCCUPANT,CLEAR,64.7,Knee-Lower Leg Foot; Entire Body; Unknown
193,20,ROSELAWN,2022-01-30 20:40:00,MALE,FATAL,ANGLE,DRY,STRAIGHT GRADE,"BLACKTOP, BITUMINOUS, ASPHALT",PASSENGER CAR,DRIVER,CLOUDY,80.0,NaN
243,17,NaN,2022-02-09 06:47:00,MALE,MINOR,NOT COLLISION BETWEEN TWO MOTOR VEHICLES IN TR...,DRY,STRAIGHT LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT",PEDESTRIAN/SKATER,PEDESTRIAN,CLEAR,44.1,Head; Back; Shoulder - Upper Arm


In [8]:
weather_dim = (
    df[['weather', 'road_condtions']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Create Weather_ID (1, 2, 3, …) and cast as string for VARCHAR(5)
weather_dim['Weather_ID'] = (weather_dim.index + 1).astype(str)

# Reorder & rename to match the SQL table
weather_dim = weather_dim.rename(columns={
    'weather': 'Weather',
    'road_condtions': 'Road_Conditions'
})[['Weather_ID', 'Weather', 'Road_Conditions']]

weather_dim.head()

,Weather_ID,Weather,Road_Conditions
0,1,RAIN,WET
1,2,CLEAR,WET
2,3,CLOUDY,DRY
3,4,CLEAR,DRY
4,5,SNOW,ICE


In [9]:
road_dim = (
    df[['road_type', 'road_surface']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Create Road_ID (1, 2, 3, …) and cast as string for VARCHAR(5)
road_dim['Road_ID'] = (road_dim.index + 1).astype(str)

# Reorder & rename to match the SQL table
road_dim = road_dim.rename(columns={
    'road_type': 'Road_Type',
    'road_surface': 'Road_Surface'
})[['Road_ID', 'Road_Type', 'Road_Surface']]
road_dim.head()

,Road_ID,Road_Type,Road_Surface
0,1,CURVE GRADE,"BLACKTOP, BITUMINOUS, ASPHALT"
1,2,CURVE LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT"
2,3,STRAIGHT LEVEL,CONCRETE
3,4,STRAIGHT LEVEL,"BLACKTOP, BITUMINOUS, ASPHALT"
4,5,STRAIGHT GRADE,"BLACKTOP, BITUMINOUS, ASPHALT"


In [10]:
# Merge df with weather_dim to get Weather_ID
df_w = df.merge(
    weather_dim,
    left_on=['weather', 'road_condtions'],
    right_on=['Weather', 'Road_Conditions']
)

# Merge with road_dim to get Road_ID
df_wr = df_w.merge(
    road_dim,
    left_on=['road_type', 'road_surface'],
    right_on=['Road_Type', 'Road_Surface']
)

crash_df = (
    df_wr[['crashdate', 'location', 'crash_type',
           'temperature_f', 'Weather_ID', 'Road_ID']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Create Crash_ID as string
crash_df['Crash_ID'] = (crash_df.index + 1).astype(str)

# Rename columns to match SQL table
crash_df = crash_df.rename(columns={
    'crashdate': 'Date',
    'crash_type': 'Crash_Type',
    'temperature_f': 'Temperature',
    'location': 'Location'
})[['Crash_ID', 'Crash_Type', 'Date',
     'Temperature', 'Location', 'Road_ID', 'Weather_ID']]

crash_df.head()


,Crash_ID,Crash_Type,Date,Temperature,Location,Road_ID,Weather_ID
0,1,ANGLE,2022-01-01 00:01:00,52.8,PLEASANT RIDGE,1,1
1,2,UNKNOWN,2022-01-01 00:37:00,52.8,CAMP WASHINGTON,2,1
2,3,NOT COLLISION BETWEEN TWO MOTOR VEHICLES IN TR...,2022-01-01 02:44:00,35.4,MOUNT AUBURN,1,1
3,4,NOT COLLISION BETWEEN TWO MOTOR VEHICLES IN TR...,2022-01-01 09:15:00,57.8,CAMP WASHINGTON,2,1
4,5,ANGLE,2022-01-01 14:28:00,53.8,DOWNTOWN,3,2


In [11]:
# 1. Attach Crash_ID to the original df 
df_with_crash = df_wr.merge(
    crash_df,
    left_on=['crashdate', 'location', 'crash_type', 'temperature_f', 'Weather_ID', 'Road_ID'],
    right_on=['Date', 'Location', 'Crash_Type', 'Temperature', 'Weather_ID', 'Road_ID'],
    how='left',
    validate='many_to_one'   # each person row -> at most one crash
)

# 2. Build Vehicle dimension/table: one row per (Crash_ID, vehicle_type)
vehicle_df = (
    df_with_crash[['Crash_ID', 'vehicle_type']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Create Vehicle_ID (string, to match your Crash_ID style)
vehicle_df['Vehicle_ID'] = (vehicle_df.index + 1).astype(str)

# Rename + reorder columns for SQL Vehicle table
vehicle_df = vehicle_df.rename(columns={
    'vehicle_type': 'Vehicle_Type'
})[['Vehicle_ID', 'Crash_ID', 'Vehicle_Type']]

vehicle_df.head()

,Vehicle_ID,Crash_ID,Vehicle_Type
0,1,1,PASSENGER CAR
1,2,2,SPORT UTILITY VEHICLE
2,3,3,PASSENGER CAR
3,4,4,PASSENGER CAR
4,5,5,SPORT UTILITY VEHICLE


In [12]:
person_df = df_with_crash.merge(
    vehicle_df,
    left_on=['Crash_ID', 'vehicle_type'],
    right_on=['Crash_ID', 'Vehicle_Type'],
    how='left',
    validate='many_to_one'   # many persons -> one vehicle
)

# Keep person-related columns + Vehicle_ID
person_df = person_df[[
    'age',
    'gender',
    'injury_area',
    'injured',
    'injury',
    'Vehicle_ID'
]]

# Add PersonID surrogate key
person_df['PersonID'] = (person_df.index + 1).astype(str)

# Rename to match SQL Person table
person_df = person_df.rename(columns={
    'age': 'Age',
    'gender': 'Gender',
    'injury_area': 'Injury_Area',
    'injured': 'Injured',
    'injury': 'Injury'
})[['PersonID', 'Age', 'Gender', 'Injury_Area', 'Injured', 'Injury', 'Vehicle_ID']]

person_df.head()

,PersonID,Age,Gender,Injury_Area,Injured,Injury,Vehicle_ID
0,1,66,MALE,Abdomen - Pelvis,DRIVER,MINOR,1
1,2,49,MALE,Abdomen - Pelvis; Neck,DRIVER,MINOR,2
2,3,9,MALE,Back,OCCUPANT,MINOR,2
3,4,39,FEMALE,Does Not Apply,OCCUPANT,MINOR,2
4,5,38,MALE,Neck; Face; Does Not Apply,DRIVER,MINOR,3


In [13]:
# Convert all NaN/NaT in these dfs to None so SQL gets real NULL --- **AI help**
for name in ["crash_df", "person_df", "weather_dim", "road_dim", "vehicle_df"]:
    df_obj = globals()[name]
    globals()[name] = df_obj.where(pd.notnull(df_obj), None)

In [14]:
#import pyodbc

# Build row lists in the same column order as SQL tables ---

weather_rows = list(
    weather_dim[['Weather_ID', 'Weather', 'Road_Conditions']]
    .itertuples(index=False, name=None)
)

road_rows = list(
    road_dim[['Road_ID', 'Road_Type', 'Road_Surface']]
    .itertuples(index=False, name=None)
)

crash_rows = list(
    crash_df[['Crash_ID', 'Crash_Type', 'Date', 'Temperature', 'Location', 'Road_ID', 'Weather_ID']]
    .itertuples(index=False, name=None)
)

vehicle_rows = list(
    vehicle_df[['Vehicle_ID', 'Vehicle_Type', 'Crash_ID']]
    .itertuples(index=False, name=None)
)

person_rows = list(
    person_df[['PersonID', 'Age', 'Gender', 'Injury_Area', 'Injured', 'Injury', 'Vehicle_ID']]
    .itertuples(index=False, name=None)
)

# INSERT statements (one per table) --- **AI help**

insert_weather_sql = """
INSERT INTO final_project3.Weather (Weather_ID, Weather, Road_Conditions)
VALUES (?, ?, ?)
"""

insert_road_sql = """
INSERT INTO final_project3.Road (Road_ID, Road_Type, Road_Surface)
VALUES (?, ?, ?)
"""

insert_crash_sql = """
INSERT INTO final_project3.Crash (Crash_ID, Crash_Type, [Date], Temperature, Location, Road_ID, Weather_ID)
VALUES (?, ?, ?, ?, ?, ?, ?)
"""

insert_vehicle_sql = """
INSERT INTO final_project3.Vehicle (Vehicle_ID, Vehicle_Type, Crash_ID)
VALUES (?, ?, ?)
"""

insert_person_sql = """
INSERT INTO final_project3.Person (PersonID, Age, Gender, Injury_Area, Injured, Injury, Vehicle_ID)
VALUES (?, ?, ?, ?, ?, ?, ?)
"""

# Execute inserts in FK-safe order: Weather/Road → Crash → Vehicle → Person ---

conn = pyodbc.connect(conn_str)
cursor = conn.cursor()
cursor.fast_executemany = True   # speeds up bulk insert


# Insert data
cursor.executemany(insert_weather_sql, weather_rows)
cursor.executemany(insert_road_sql,    road_rows)
cursor.executemany(insert_crash_sql,   crash_rows)
cursor.executemany(insert_vehicle_sql, vehicle_rows)
cursor.executemany(insert_person_sql,  person_rows)

conn.commit()
conn.close()
print("All data inserted successfully into Weather, Road, Crash, Vehicle, Person.")


All data inserted successfully into Weather, Road, Crash, Vehicle, Person.


In [15]:
person_df.head()

,PersonID,Age,Gender,Injury_Area,Injured,Injury,Vehicle_ID
0,1,66,MALE,Abdomen - Pelvis,DRIVER,MINOR,1
1,2,49,MALE,Abdomen - Pelvis; Neck,DRIVER,MINOR,2
2,3,9,MALE,Back,OCCUPANT,MINOR,2
3,4,39,FEMALE,Does Not Apply,OCCUPANT,MINOR,2
4,5,38,MALE,Neck; Face; Does Not Apply,DRIVER,MINOR,3
